# CITE-seq BMMC TAD co-regulation analysis

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import glob
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
import scipy.stats
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig4_neurips2021_cite_bmmc_fc_interaction_gene_layer")
os.makedirs(_OUTDIR_ABS, exist_ok=True)
PRIMARY = "../.experiment_data/primary"
CHIP = "../.experiment_data/reruns/fig4_tad/chip"
RESOURCES = "../preprocessing/resources"
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

In [ ]:
SEEDS = [11, 13, 14]
AUROC, HELDOUT = "predictors_label_valid_roc", "predictors_label_test_roc"
FOLDER = {"go": "calc_go_terms_cell_type", "genes": "calc_genes_cell_type", "groups": "calc_gene_groups_cell_type"}

networkvi_tad_paths = {
    11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s11/default",
    13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s13/default",
    14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s14/default",
}

networkvi_no_tad_paths = {
    11: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s11/default",
    13: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s13/default",
    14: "../.experiment_data/reruns/fig4_tad/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s14/default",
}


def run_dir(arch, seed, tad_map="ENCFF173VDJ"):
    if arch == "notad":
        return networkvi_no_tad_paths[seed]
    if tad_map == "ENCFF173VDJ":
        return networkvi_tad_paths[seed]
    raise KeyError(tad_map)


def cell_type_key(label):
    return label.replace("/", "_").replace(" ", "_")


def nice(ct):
    return ct.replace("_", " ").replace("G M prog", "G/M prog").replace("MK E prog", "MK/E prog")


def load_importances(directory, kind, modality="expression"):
    out = {}
    for f in sorted(glob.glob(f"{directory}/{FOLDER[kind]}/{modality}_cell_type_vs_rest_*_evaluated_graph_csv_with_ci.pkl")):
        label = os.path.basename(f).split("_vs_rest_", 1)[1].split("_res_unp_act_mask")[0]
        out[cell_type_key(label)] = pd.read_pickle(f).apply(pd.to_numeric, errors="coerce")
    return out


def paired(tad, base, column=AUROC):
    return {ct: pd.concat([tad[ct][column].rename("tad"), base[ct][column].rename("base")], axis=1, join="inner") for ct in sorted(set(tad) & set(base))}


def pct(x):
    return 100 * (np.asarray(x) - 1)


def boot_ci(x):
    r = scipy.stats.bootstrap((np.asarray(x, dtype=float),), np.mean, n_resamples=10000, confidence_level=0.95, random_state=0)
    return r.confidence_interval.low, r.confidence_interval.high


def averaged(tables):
    cts = sorted(set.intersection(*[set(t) for t in tables]))
    return {ct: pd.concat([t[ct][AUROC] for t in tables], axis=1, join="inner").mean(axis=1).rename(AUROC).to_frame() for ct in cts}


def gain_ci(ratios):
    lo, hi = boot_ci(ratios)
    return {"value (%)": float(pct(np.mean(ratios))), "95% CI low (%)": float(pct(lo)), "95% CI high (%)": float(pct(hi)), "Wilcoxon p": wilcoxon(ratios, np.ones_like(ratios)).pvalue}

In [ ]:
gene_groups_importances = {}
for modality in ["expression", "protein"]:
    for s in SEEDS:
        imp = load_importances(run_dir("tad", s), "groups", modality)
        gene_groups_importances[(modality, s)] = pd.DataFrame({ct: imp[ct][AUROC] for ct in sorted(imp, key=nice)})
{k: v.shape for k, v in gene_groups_importances.items()}

# Gene groups importance statistics

In [ ]:
pd.DataFrame({f"{modality} s{s}": {f"> {th}": int((gene_groups_importances[(modality, s)].values > th).sum()) for th in [0.6, 0.65, 0.9]} for modality in ["expression", "protein"] for s in SEEDS})

In [ ]:
for modality in ["expression", "protein"]:
    mean_importance = sum(gene_groups_importances[(modality, s)] for s in SEEDS) / len(SEEDS)
    set_figure_params("double", 9)
    fig, axes = plt.subplots(9, 5)
    for ax, ct in zip(axes.flatten(), mean_importance.columns):
        ax.hist(mean_importance[ct].dropna(), bins=50)
        ax.set_title(nice(ct))
        ax.set_xlabel("TAD importance")
        ax.set_ylabel("prevalence")
        ax.set_xlim(right=1.0)
    plt.tight_layout()
    plt.show()